# Notebook 12 — LayoutLMv3 Fine-Tuning

Fine-tunes `microsoft/layoutlmv3-base` for token classification on the
FATURA invoice field extraction task prepared in Notebook 11.

**Model:** `microsoft/layoutlmv3-base`  
**Task:** Token classification (NER) — 13 BIO labels  
**Fields:** INVOICE_NUMBER, INVOICE_DATE, DUE_DATE, ISSUER_NAME, RECIPIENT_NAME, TOTAL_AMOUNT  
**Evaluation:** entity-level F1 via `seqeval`  
**Early stopping:** patience 3 on val entity F1  

> **No generative AI** — LayoutLMv3 is a discriminative encoder-only model.

## 0. Dependencies

In [1]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg_name, import_name in [('datasets', 'datasets'), ('seqeval', 'seqeval')]:
    try:
        __import__(import_name)
        print(f'{pkg_name} already installed')
    except ImportError:
        print(f'Installing {pkg_name}…')
        pip_install(pkg_name)
        print(f'{pkg_name} installed')

/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sofiiaavetisian/Desktop/UNI/statistical_learning/FINAL_PROJECT/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


datasets already installed
seqeval already installed


## 1. Imports & configuration

In [ ]:
import time
t0 = time.time()
from transformers import LayoutLMv3ForTokenClassification, LayoutLMv3Processor
print("import_ok_sec:", round(time.time() - t0, 2))


In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image
print(f'PIL version: {Image.__version__}')
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
print(f'Torchvision version: {torch.__version__}')
from transformers import (
    LayoutLMv3ForTokenClassification,
    LayoutLMv3Processor,
)
print(f'Transformers version: {torch.__version__}')
from datasets import load_from_disk
from seqeval.metrics import classification_report as seq_report, f1_score as seq_f1
print (f'PyTorch version: {torch.__version__}')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path('..').resolve()
DATASET_DIR   = PROJECT_ROOT / 'data' / 'processed' / 'layoutlmv3_dataset'
MODEL_OUT_DIR = PROJECT_ROOT / 'models' / 'experimental' / 'layoutlmv3_fatura'
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────────────────────
BASE_MODEL   = 'microsoft/layoutlmv3-base'
EPOCHS       = 10
LR           = 1e-5
WEIGHT_DECAY = 0.01
BATCH_SIZE   = 2    # LayoutLMv3 is memory-heavy; keep batch small
PATIENCE     = 3    # early-stopping patience (val entity F1)
MAX_LENGTH   = 512

DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                       'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Dataset dir: {DATASET_DIR}')
print(f'Model output: {MODEL_OUT_DIR}')

PIL version: 11.3.0
Torchvision version: 2.8.0


## 2. Load label schema & dataset

In [ ]:
with open(DATASET_DIR / 'label2id.json') as f:
    label2id = json.load(f)
with open(DATASET_DIR / 'id2label.json') as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

NUM_LABELS = len(label2id)
BIO_LABELS = [id2label[i] for i in range(NUM_LABELS)]

print(f'Labels ({NUM_LABELS}):', BIO_LABELS)

raw_dataset = load_from_disk(str(DATASET_DIR))
print('\nDataset loaded:')
print(raw_dataset)

## 3. Processor & tokenisation

In [ ]:
processor = LayoutLMv3Processor.from_pretrained(BASE_MODEL, apply_ocr=False)
# apply_ocr=False because we supply words & bboxes from our HF-format annotations
print('Processor loaded')

In [ ]:
def encode_example(example):
    """
    Tokenise a single record for LayoutLMv3.

    LayoutLMv3 uses a word-piece tokeniser: one word may split into multiple
    sub-tokens.  We propagate the word-level label to the first sub-token and
    use -100 (PyTorch ignore index) for continuation sub-tokens so they do not
    contribute to the loss.
    """
    image    = Image.open(example['image_path']).convert('RGB')
    words    = example['words']
    bboxes   = example['bboxes']   # already normalised to [0, 1000]
    word_labels = example['ner_tags']  # one per word

    encoding = processor(
        image,
        words,
        boxes=bboxes,
        word_labels=word_labels,
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
        return_tensors='pt',
    )
    # Squeeze batch dim — DataLoader will re-add it
    return {k: v.squeeze(0) for k, v in encoding.items()}


# Sanity check
sample_enc = encode_example(raw_dataset['train'][0])
print('Encoding keys:', list(sample_enc.keys()))
print('input_ids shape    :', sample_enc['input_ids'].shape)
print('attention_mask shape:', sample_enc['attention_mask'].shape)
print('bbox shape         :', sample_enc['bbox'].shape)
print('labels shape       :', sample_enc['labels'].shape)

## 4. PyTorch Dataset & DataLoaders

In [ ]:
class FaturaTokenDataset(Dataset):
    """Wraps a HuggingFace Dataset split and applies LayoutLMv3 encoding on-the-fly."""

    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return encode_example(self.data[idx])


train_ds = FaturaTokenDataset(raw_dataset['train'])
val_ds   = FaturaTokenDataset(raw_dataset['val'])
test_ds  = FaturaTokenDataset(raw_dataset['test'])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches  : {len(val_loader)}')
print(f'Test batches : {len(test_loader)}')

## 5. Model initialisation

In [ ]:
model = LayoutLMv3ForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,   # classification head is new
)
model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params   : {total_params:,}')
print(f'Trainable params: {trainable:,}')

## 6. Evaluation helper (seqeval entity-level F1)

In [ ]:
def evaluate(model, loader):
    """
    Run inference over a DataLoader and compute seqeval entity-level F1.

    Returns (entity_f1: float, per_class_report: str).
    Tokens with label == -100 (padding / sub-token continuations) are ignored.
    """
    model.eval()
    all_preds, all_trues = [], []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            logits  = outputs.logits  # (B, seq_len, num_labels)
            preds   = logits.argmax(dim=-1)  # (B, seq_len)
            labels  = batch['labels']        # (B, seq_len)

            for pred_seq, true_seq in zip(preds, labels):
                pred_labels, true_labels = [], []
                for p, t in zip(pred_seq, true_seq):
                    if t.item() == -100:      # ignore padding / sub-tokens
                        continue
                    pred_labels.append(id2label[p.item()])
                    true_labels.append(id2label[t.item()])
                all_preds.append(pred_labels)
                all_trues.append(true_labels)

    entity_f1 = seq_f1(all_trues, all_preds)
    report    = seq_report(all_trues, all_preds, digits=4)
    return entity_f1, report

## 7. Training loop with early stopping

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_val_f1   = -1.0
best_epoch    = -1
patience_left = PATIENCE
history       = []     # list of dicts, one per epoch

CKPT_PATH = MODEL_OUT_DIR / 'best_checkpoint'

for epoch in range(1, EPOCHS + 1):
    # ── Training ───────────────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss    = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        if step % 100 == 0 or step == len(train_loader):
            print(f'  Epoch {epoch}/{EPOCHS} | Step {step}/{len(train_loader)} '
                  f'| loss={loss.item():.4f}', end='\r')

    avg_loss = total_loss / len(train_loader)

    # ── Validation ─────────────────────────────────────────────────────────
    val_f1, val_report = evaluate(model, val_loader)

    rec = {'epoch': epoch, 'train_loss': avg_loss, 'val_entity_f1': val_f1}
    history.append(rec)
    print(f'\nEpoch {epoch}/{EPOCHS} — train_loss={avg_loss:.4f}  val_entity_f1={val_f1:.4f}')

    # ── Early stopping & checkpoint ────────────────────────────────────────
    if val_f1 > best_val_f1:
        best_val_f1   = val_f1
        best_epoch    = epoch
        patience_left = PATIENCE
        model.save_pretrained(str(CKPT_PATH))
        processor.save_pretrained(str(CKPT_PATH))
        print(f'  ✓ New best val entity F1 = {best_val_f1:.4f} — checkpoint saved')
    else:
        patience_left -= 1
        print(f'  No improvement. Patience left: {patience_left}/{PATIENCE}')
        if patience_left == 0:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nBest val entity F1 = {best_val_f1:.4f} at epoch {best_epoch}')

## 8. Test-set evaluation

In [ ]:
# Load best checkpoint for test evaluation
best_model = LayoutLMv3ForTokenClassification.from_pretrained(
    str(CKPT_PATH),
    id2label=id2label,
    label2id=label2id,
)
best_model.to(DEVICE)

test_f1, test_report = evaluate(best_model, test_loader)
print(f'Test entity F1 (best checkpoint): {test_f1:.4f}')
print('\nPer-entity classification report:')
print(test_report)

## 9. Training history plot

In [ ]:
import matplotlib.pyplot as plt

epochs_done = [r['epoch'] for r in history]
train_losses = [r['train_loss'] for r in history]
val_f1s      = [r['val_entity_f1'] for r in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_done, train_losses, marker='o', color='steelblue')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_done, val_f1s, marker='o', color='darkorange')
ax2.axvline(best_epoch, linestyle='--', color='red', label=f'Best epoch {best_epoch}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Entity F1'); ax2.set_title('Val Entity F1')
ax2.legend(); ax2.grid(True, alpha=0.3)

fig.tight_layout()
plot_path = MODEL_OUT_DIR / 'training_history.png'
fig.savefig(plot_path, dpi=130, bbox_inches='tight')
plt.show()
print('Plot saved to:', plot_path)

## 10. Quick inference on 3 test examples

In [ ]:
def extract_fields_from_encoding(words, pred_ids, id2label):
    """
    Given word-level predictions, group consecutive same-field tokens
    and return a dict of {field_name: extracted_text}.
    """
    fields = {}
    current_field, current_tokens = None, []

    for word, label_id in zip(words, pred_ids):
        label = id2label[label_id]
        if label.startswith('B-'):
            if current_field:
                fields[current_field] = ' '.join(current_tokens)
            current_field  = label[2:]
            current_tokens = [word]
        elif label.startswith('I-') and current_field == label[2:]:
            current_tokens.append(word)
        else:
            if current_field:
                fields[current_field] = ' '.join(current_tokens)
            current_field, current_tokens = None, []

    if current_field:
        fields[current_field] = ' '.join(current_tokens)

    return fields


best_model.eval()
processor_loaded = LayoutLMv3Processor.from_pretrained(str(CKPT_PATH), apply_ocr=False)

for i in range(3):
    example    = raw_dataset['test'][i]
    image      = Image.open(example['image_path']).convert('RGB')
    words      = example['words']
    bboxes     = example['bboxes']

    encoding = processor_loaded(
        image, words, boxes=bboxes,
        truncation=True, padding='max_length',
        max_length=MAX_LENGTH, return_tensors='pt',
    )
    encoding = {k: v.to(DEVICE) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = best_model(**encoding)
    preds = outputs.logits.argmax(-1).squeeze(0).cpu().tolist()

    # Align predictions back to word level using token offsets
    word_ids = encoding.get('overflow_to_sample_mapping', None)
    # Use labels mask: positions where input_ids are not padding
    token_preds = preds[:len(words)]  # simple truncation to word count

    extracted = extract_fields_from_encoding(words, token_preds, id2label)

    print(f'\n=== Test example {i} ({Path(example["image_path"]).stem}) ===')
    for field, value in extracted.items():
        print(f'  {field:<20}: {value}')

## 11. Save training config & metrics

In [ ]:
config = {
    'base_model':       BASE_MODEL,
    'num_labels':       NUM_LABELS,
    'labels':           BIO_LABELS,
    'epochs_run':       len(history),
    'best_epoch':       best_epoch,
    'best_val_entity_f1': round(best_val_f1, 6),
    'test_entity_f1':   round(test_f1, 6),
    'lr':               LR,
    'weight_decay':     WEIGHT_DECAY,
    'batch_size':       BATCH_SIZE,
    'max_length':       MAX_LENGTH,
    'patience':         PATIENCE,
    'training_data':    'FATURA synthetic invoices (Zenodo 8261508) — layoutlm_HF_format annotations',
    'note':             'Discriminative token-classifier only; no generative AI.',
}

config_path = MODEL_OUT_DIR / 'layoutlmv3_training_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

# Also save full epoch history
history_path = MODEL_OUT_DIR / 'training_history.json'
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)

print('Saved training config to:', config_path)
print('Saved training history to:', history_path)
print()
print('=' * 60)
print('NOTEBOOK 12 — TRAINING COMPLETE')
print('=' * 60)
print(f'Best val entity F1 : {best_val_f1:.4f}  (epoch {best_epoch})')
print(f'Test entity F1     : {test_f1:.4f}')
print(f'Checkpoint saved to: {CKPT_PATH}')
print()
print('Next: Notebook 13 — Full pipeline inference (demo)')